# Toy sanity check: does the flow recover `y = 2x + N(0, 0.15)`?

Two variables, one linear-plus-gaussian relationship, a small affine flow. If the
sampler works at all, the fitted cloud should be a straight band: slope 2, residual
sd 0.15, residual independent of `x`.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")   # tiny problem, CPU is fine
import numpy as np
import matplotlib.pyplot as plt
from calibrated_response.models.variable import ContinuousVariable
from calibrated_response.models.natural_response import parse_natural_syntax as P
from calibrated_response.maxent_sampler.distribution_builder import DistributionBuilder

SLOPE, NOISE = 2.0, 0.15

In [ ]:
variables = [
    ContinuousVariable(name="x", description="predictor", lower_bound=0.0, upper_bound=1.0),
    ContinuousVariable(name="y", description="response",  lower_bound=-0.6, upper_bound=2.6),
]
estimates = [P(e) for e in [
    "E[x] = 0.5",              # x ~ uniform on its box (maxent default)
    "y = 2*x ~ N(0, 0.15)",   # the linear + gaussian relationship
]]

b = DistributionBuilder(variables, estimates, n_layers=6, hidden=64)
b.build(steps=1500, lr=3e-3, n_samples=2048, seed=0)
print("done — joint entropy (nats):", round(b.entropy(), 3))

In [ ]:
d = b.sample_dict(40000, seed=1)
x, y = d["x"], d["y"]
r = y - SLOPE * x
slope_hat, intercept_hat = np.polyfit(x, y, 1)

print(f"x mean/std        : {x.mean():.3f} / {x.std():.3f}   (uniform: 0.500 / 0.289)")
print(f"recovered slope   : {slope_hat:.3f}     (true {SLOPE})")
print(f"recovered intercept: {intercept_hat:+.3f}    (true 0)")
print(f"residual mean/std : {r.mean():+.3f} / {r.std():.3f}   (true 0 / {NOISE})")
print(f"corr(x, residual) : {np.corrcoef(x, r)[0,1]:+.3f}   (want ~0)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

# 1. joint cloud vs the true line
ax[0].scatter(x[:4000], y[:4000], s=3, alpha=0.15)
xs = np.linspace(0, 1, 100)
ax[0].plot(xs, SLOPE * xs, "r-", lw=2, label="y = 2x")
ax[0].set(xlabel="x", ylabel="y", title="fitted joint"); ax[0].legend()

# 2. residual marginal vs N(0, 0.15)
ax[1].hist(r, bins=80, density=True, alpha=0.6)
rr = np.linspace(-4 * NOISE, 4 * NOISE, 200)
ax[1].plot(rr, np.exp(-rr**2 / (2 * NOISE**2)) / (NOISE * np.sqrt(2 * np.pi)), "r-", lw=2)
ax[1].set(xlabel="residual y - 2x", title=f"residual vs N(0, {NOISE})")

# 3. E[y|x] and +/-2sd band track the truth across x
bins = np.linspace(0, 1, 15)
idx = np.digitize(x, bins)
mid = 0.5 * (bins[:-1] + bins[1:])
mean = [y[idx == i].mean() for i in range(1, len(bins))]
sd = [y[idx == i].std() for i in range(1, len(bins))]
ax[2].plot(xs, SLOPE * xs, "r-", lw=1, label="true mean")
ax[2].errorbar(mid, mean, yerr=2 * np.array(sd), fmt="o", capsize=3, label="fitted E[y|x] +/-2sd")
ax[2].set(xlabel="x", ylabel="y", title="conditional"); ax[2].legend()

plt.tight_layout()